<a href="https://colab.research.google.com/github/repulsivityy/learning-LLMs/blob/main/notebooks/00_mechanics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Initial Setup

Setting up the colab

In [ ]:
import os

REPO_URL = "https://github.com/repulsivityy/learning-LLMs.git"
REPO_DIR = "/content/learning-LLMs"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/repulsivityy/learning-LLMs.git

In [ ]:
!pip install -q torch numpy
!mkdir -p notebooks src

In [ ]:
%%writefile src/tokenizer.py
from collections import defaultdict


def get_pair_counts(word_freqs):
    pair_counts = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += freq
    return pair_counts


def merge_pair(pair, word_freqs):
    new_word_freqs = {}
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs


def train_bpe(word_freqs, num_merges):
    word_freqs = dict(word_freqs)
    merges = []
    for step in range(num_merges):
        pair_counts = get_pair_counts(word_freqs)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=pair_counts.get)
        word_freqs = merge_pair(best_pair, word_freqs)
        merges.append(best_pair)
        print(f"Merge {step + 1}: {best_pair}  (count={pair_counts[best_pair]})")
    return word_freqs, merges

## **Tokenization: Byte-Pair Encoding (BPE)**

**Why it matters:** a language model never sees text — only a sequence of integers.
Tokenization is the text ↔ integer conversion. BPE is the algorithm nearly every
modern LLM uses to decide what those "chunks" should be.

**ELI5:** imagine a box of individual letter tiles. Every time two tiles keep
turning up next to each other ("t" + "h" in "the", "this", "that"), you glue
them into one bigger tile. Do that thousands of times and your box ends up
with tiles for whole common words, plus loose letters for anything unusual —
so you're never stuck on a word you've never seen.

In [ ]:
import sys
sys.path.append('/content/learning-LLMs/src')

from tokenizer import get_pair_counts, merge_pair, train_bpe

### Toy corpus

The classic example from the original BPE paper (Sennrich et al., 2016) —
useful because we know exactly what the "correct" output should be, so we
can check our implementation against it.

Each word is a tuple of characters plus an end-of-word marker `</w>`, mapped
to how many times it appears in the corpus.

In [ ]:
word_freqs = {
    ('l', 'o', 'w', '</w>'): 5,
    ('l', 'o', 'w', 'e', 'r', '</w>'): 2,
    ('n', 'e', 'w', 'e', 's', 't', '</w>'): 6,
    ('w', 'i', 'd', 'e', 's', 't', '</w>'): 3,
}
word_freqs

### Step 1 — `get_pair_counts`

Counts every adjacent symbol pair across the corpus, weighted by how often
each word appears.

**ELI5:** for every word, look at every two touching letters, and add "how
many times this whole word shows up" to that pair's running total.

In [ ]:
pair_counts = get_pair_counts(word_freqs)
sorted(pair_counts.items(), key=lambda x: -x[1])[:5]

### Step 2 — `merge_pair`

Fuses the winning pair everywhere it occurs.

**ELI5:** walk through each word letter by letter; every time you spot the
exact pair you're gluing, weld those two into one tile and hop over both.

In [ ]:
merged = merge_pair(('e', 's'), word_freqs)
merged

### Step 3 — `train_bpe`: the full training loop

Repeats "find the most frequent pair → merge it" for a fixed number of
steps. The **ordered list of merges is the entire trained tokenizer** — to
tokenize new text later, you replay these same merges in this same order.

In [ ]:
final_word_freqs, merges = train_bpe(word_freqs, num_merges=8)
merges

In [ ]:
final_word_freqs

### Step 4 — `encode` / `decode`

So far we've only *trained* the tokenizer (learned the merge list). Now we
use it: take brand-new text, and replay the learned merges — in the exact
order we learned them — to turn it into tokens.

**ELI5:** you already decided the gluing order during training (glue "es"
first, then "est", then "low", etc.). Now, for any new word, you just apply
those same glue steps in that same order and see what tiles you end up with.
Words you've never seen still work — they just get broken into whatever
familiar pieces are left over.

In [ ]:
%%writefile src/tokenizer.py
from collections import defaultdict


def get_pair_counts(word_freqs):
    pair_counts = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += freq
    return pair_counts


def merge_pair(pair, word_freqs):
    new_word_freqs = {}
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs


def train_bpe(word_freqs, num_merges):
    word_freqs = dict(word_freqs)
    merges = []
    for step in range(num_merges):
        pair_counts = get_pair_counts(word_freqs)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=pair_counts.get)
        word_freqs = merge_pair(best_pair, word_freqs)
        merges.append(best_pair)
        print(f"Merge {step + 1}: {best_pair}  (count={pair_counts[best_pair]})")
    return word_freqs, merges


def get_word_tokens(word, merges):
    """Apply a learned merge list, in order, to a single word."""
    tokens = list(word) + ['</w>']
    for pair in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(tokens[i] + tokens[i + 1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


def encode(text, merges):
    """Whitespace-split text into words, then BPE-tokenize each word."""
    all_tokens = []
    for word in text.strip().split():
        all_tokens.extend(get_word_tokens(word, merges))
    return all_tokens


def decode(tokens):
    """Join tokens back into text."""
    text = ''.join(tokens).replace('</w>', ' ')
    return text.strip()

In [ ]:
import importlib
import tokenizer
importlib.reload(tokenizer)
from tokenizer import get_pair_counts, merge_pair, train_bpe, encode, decode

### Try it on a word the tokenizer never saw

`"lowest"` never appeared in training — only `"low"`, `"lower"`, `"newest"`,
`"widest"` did. Watch what happens.

In [ ]:
tokens = encode("lowest", merges)
tokens

In [ ]:
decode(tokens)

### Try it yourself!

As expected, it shows up as
>"low", "est</w>"

Try with other words:
- highest (not part of the training, so it should only show h,i,g,h,est)
- test (t,est)
- or something outrageous like zebra (z,e,b,r,a)

## **Attention**

**Why it matters:** so far every word/token exists on its own — the model
has no way to relate them to each other. Attention is the mechanism that
lets each token look at every other token and pull in the information it
actually needs from them.

**Technical:** for each token you produce three vectors — a **Query** (what
am I looking for), a **Key** (what do I contain, for others to match
against), and a **Value** (what information do I actually offer). You
compare every Query to every Key to get relevance scores, turn those into
percentages (softmax), then blend the Value vectors using those percentages.

**ELI5:** imagine everyone in a room holding up a sign describing what they
know (Key) and a sign describing what they're curious about (Query). Every
person compares their "curious about" sign to everyone else's "what I know"
sign, gets a relevance score for each person, turns those into percentages
that add to 100%, then walks away with a blend of everyone's actual
information (Value), weighted by how relevant they were.

In [ ]:
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

**Line by line:**

- ```Q @ K.transpose(-2, -1)``` — dot product of every query against every key → a matrix where entry ```(i, j)``` says "how relevant is token ```j``` to token ```i```."

- ```/ (d_k ** 0.5)``` — the "scaled" part of "scaled dot-product attention." Without this, scores get large as the vector dimension grows, which pushes softmax into being overconfident (nearly all weight on one token) and makes training unstable.

- ```F.softmax(scores, dim=-1)``` — turns each row of raw scores into percentages that sum to 1 — "how much attention should token ```i``` pay to each other token."

- ```weights @ V``` — the actual payoff: token ```i```'s new representation is a weighted blend of everyone's Value vectors, weighted by those percentages.

### Try it on toy data

4 toy "tokens," each an 8-dimensional random vector standing in for Q, K, V
(no learned weights yet — we're just testing the mechanism itself).

In [ ]:
torch.manual_seed(0)
seq_len, d_k = 4, 8
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print(weights)
print(weights.sum(dim=-1))  # sanity check: every row should sum to 1.0

## **Multi-Head Attention**

**Why it matters:** single-head attention forces one relevance pattern per
token. Real language has multiple *kinds* of relationships at once (which
word is the subject, which word is nearby, which word rhymes...). Multi-head
runs several smaller attentions in parallel so each "head" can specialize.

This also introduces something we skipped earlier: **learned projections**.
Until now we fed raw Q/K/V directly into the attention formula. Real
transformers compute Q, K, V *from* the input embeddings via learned weight
matrices — that's what actually gets trained.

**ELI5:** instead of one person studying the whole room and forming one
opinion, split into several smaller specialist committees. Each committee
only sees its own slice of the information and forms its own opinion (its
own attention pattern). At the end, gather every committee's opinion and
merge them into one final report.

In [ ]:
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        # split d_model into (num_heads, head_dim), move heads next to batch
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # reuse the exact function from single-head attention, unmodified
        output, weights = scaled_dot_product_attention(Q, K, V)

        # merge heads back into one d_model-sized vector per token
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        return self.W_o(output), weights

#### Walking through it:

- ```self.W_q = nn.Linear(d_model, d_model)``` — this is the learned projection we skipped before. ```nn.Linear``` internally does ```x @ W.T + b```, with ```W``` and ```b``` as trainable parameters. Instead of you handing in ```Q``` directly, the model now learns how to compute ```Q``` from the raw input embeddings.

- ```Q.view(batch_size, seq_len, self.num_heads, self.head_dim)``` — chops each token's ```d_model```-sized vector into ```num_heads``` smaller chunks of size ```head_dim```. E.g. ```d_model=8, num_heads=2``` → two chunks of size 4 per token.

- ```.transpose(1, 2)``` — swaps the ```seq_len``` and ```num_heads``` dimensions, giving shape ```(batch, num_heads, seq_len, head_dim)```. This puts ```num_heads``` next to ```batch```, so from the attention function's point of view, heads just look like extra batch entries.

- ```scaled_dot_product_attention(Q, K, V)``` is reused completely unchanged. This is the payoff of using ```.transpose(-2, -1)``` instead of ```.T``` back when we wrote it — that function doesn't care how many leading dimensions ```(batch, num_heads)``` there are, it only ever touches the last two. That design decision from Lesson 1 is what makes this work for free.

- After attention, ```output.transpose(1, 2).contiguous().view(...)``` reverses the split: puts ```seq_len``` back before ```num_heads```, then flattens ```(num_heads, head_dim``` back into one ```d_model```-sized vector per token — this is the "concatenate the heads" step.

- ```self.W_o(output)``` — one final learned linear layer that mixes information across heads together, so the model can combine what each specialist committee found.

### Try it

`d_model=8`, `num_heads=2` → each head works with 4 dimensions. Check that
the output shape matches the input shape, and that the two heads produce
different attention patterns from the same input.

In [ ]:
torch.manual_seed(0)
batch_size, seq_len, d_model, num_heads = 1, 4, 8, 2

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)

output, weights = mha(x)
print("output shape:", output.shape)    # expect (1, 4, 8) — same as input
print("weights shape:", weights.shape)  # expect (1, 2, 4, 4) — one 4x4 matrix per head

In [ ]:
print("head 0 attention:\n", weights[0, 0])
print("head 1 attention:\n", weights[0, 1])

## **Residual Connection + Layer Norm**

**Why it matters:** stack raw sublayers (like attention) directly on top of
each other and gradients vanish or explode a few layers in — deep networks
just don't train. Two fixes, always used together:

- **Residual connection:** add the sublayer's output *back onto* its input,
  instead of replacing it (`x + Sublayer(x)`). This guarantees there's
  always a direct path for information (and gradients) to flow through,
  even if a layer doesn't have anything useful to add.
- **Layer norm:** after adding, rescale each token's vector so it has mean
  0 and variance 1 (then apply a small learned scale/shift). Keeps numbers
  from growing or shrinking uncontrollably as they pass through many layers.

**ELI5:** think of an airport moving walkway. You can just stand still and
still get carried forward (the residual connection carries your original
information along unchanged) — or you can also walk while on it, adding
something extra (that's what the sublayer contributes). Nothing is ever
lost by default. Layer norm is like a volume knob that gets checked after
every walkway, making sure the signal doesn't get too loud or too quiet
before the next one.

In [ ]:
class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        return self.norm(x + sublayer_output)

- ```x + sublayer_output``` — the residual connection. ```x``` is the original input to the sublayer (e.g. the embeddings going into attention), ```sublayer_output``` is what attention produced. Adding them means attention only needs to learn a correction to add on top of ```x```, not reconstruct all of ```x```'s information from scratch.

- ```nn.LayerNorm(d_model)``` — normalizes across the last dimension (size ```d_model```) independently for each token. Unlike batch norm, it doesn't look at other examples in the batch at all — just this one token's own ```d_model``` numbers — which is exactly why it works fine even with variable sequence lengths or a batch size of 1.

### Try it: attention + residual + norm together

In [ ]:
torch.manual_seed(0)
d_model, num_heads = 8, 2
mha = MultiHeadAttention(d_model, num_heads)
add_norm = ResidualLayerNorm(d_model)

x = torch.randn(1, 4, d_model)
attn_output, _ = mha(x)
output = add_norm(x, attn_output)

print("output shape:", output.shape)      # expect (1, 4, 8)
print("per-token mean:", output.mean(dim=-1))  # expect ~0 for each token
print("per-token std:", output.std(dim=-1))    # expect ~1 for each token
print("per-token std unbiased", output.std(dim=-1, unbiased=False)) # removes different defs of "std" - pytorch uses bessel's correction, but nn.Laynorm used the biased/population variance (divides by N) - see note below

### Note

When I first ran it, I got this, when we are expecting the STD to be [1, 1, 1, 1], but I got this instead.
```
output shape: torch.Size([1, 4, 8])
per-token mean: tensor([[2.9802e-08, 2.9802e-08, 0.0000e+00, 1.1176e-08]],
       grad_fn=<MeanBackward1>)
per-token std: tensor([[1.0690, 1.0690, 1.0690, 1.0690]], grad_fn=<StdBackward0>)
```

After asking Claude, I got this answer below:


The std being ```1.0690``` instead of exactly ```1.0``` isn't an error — it's a subtle mismatch between two different definitions of "standard deviation": PyTorch's ```.std(```) uses **Bessel's correction** by default (divides by N-1), but ```nn.LayerNorm``` internally normalizes using the **biased/population variance (divides by N)**. With ```d_model=8``` features, the correction factor works out to exactly ```√(8/7) ≈ 1.0690``` — matching your number precisely. That's not a coincidence; it's the whole discrepancy.

Try this to see it line up:

```print(output.std(dim=-1, unbiased=False))```

That should print ```~1.0``` for every token — confirming ```LayerNorm``` really did normalize to unit variance, just measured the way it actually computes it internally, not the way ```.std()``` measures by default.

Small detail, but worth internalizing: it's the kind of "why doesn't this match the textbook number exactly" question that trips people up later when debugging real training runs — the model isn't broken, you're just comparing two slightly different statistics. Run that and confirm, then we'll move to the feed-forward sublayer.

## **Feed-Forward Sublayer**

**Why it matters:** attention's whole job is letting tokens exchange
information with each other. The feed-forward layer does the opposite —
it processes each token *independently*, with the exact same set of
weights applied at every position. This is where a lot of a model's actual
learned "knowledge" ends up stored.

Structure: expand → non-linearity → shrink back down.
`Linear(d_model → d_ff) → activation → Linear(d_ff → d_model)`.
`d_ff` is usually much wider than `d_model` (4x in the original paper) —
extra room for the network to compute in.

**ELI5:** after everyone in the room shared and blended notes with each
other (attention), each person now goes off privately and thinks it over
using their own notebook — same thinking process for everyone, but each
person applies it only to their own (now-updated) notes.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.activation(self.linear1(x)))

- ```linear1```: expands each token's vector from ```d_model``` to the wider ```d_ff```, giving the network more room to compute.

- ```self.activation (ReLU, max(0, x))```: the crucial non-linearity. Without it, ```linear2(linear1(x))``` would just collapse into one big linear transform — stacking two linear layers with nothing between them buys you nothing. ReLU is what lets this learn actually non-linear functions.

- ```linear2```: projects back down to ```d_model```, so the shape matches the input — required, since we're about to add it back via a residual connection.

### Putting it all together: one full Transformer block

Attention (mix information across tokens) → add & norm → feed-forward
(process each token independently) → add & norm. This is one repeatable
"layer" — real models just stack many of these.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model) # First for Attention
        self.ff = FeedForward(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model) # Next for Feed-Forward

    def forward(self, x):
        attn_output, weights = self.mha(x)
        x = self.add_norm1(x, attn_output)

        ff_output = self.ff(x)
        x = self.add_norm2(x, ff_output)

        return x, weights

In [ ]:
torch.manual_seed(0)
d_model, num_heads, d_ff = 8, 2, 32
block = TransformerBlock(d_model, num_heads, d_ff)

x = torch.randn(1, 4, d_model)
output, weights = block(x)
print("output shape:", output.shape)  # expect (1, 4, 8) — same as input

## **KV-Cache**

**Why it matters:** when generating text one token at a time, a naive
implementation recomputes Key and Value vectors for *every* token seen so
far, at *every* single step — even though old tokens' K/V never change once
computed. A KV-cache just remembers them.

**ELI5:** imagine summarizing a conversation that keeps growing one
sentence at a time. The naive way: every time someone adds a sentence, you
re-read and re-summarize the *entire* conversation from scratch. The cached
way: you keep your running notes, and only process the brand-new sentence,
appending it to what you already wrote down.

**The cost difference:** generating `n` tokens naively means recomputing
K/V for `1 + 2 + 3 + ... + n = n(n+1)/2` token-equivalents — quadratic
growth. With a cache, it's exactly `n` — computed once each, ever. At
`n=1000`, that's ~500,500 vs. 1,000. This is why every real LLM inference
system uses a KV-cache.

In [ ]:
class MultiHeadAttentionWithCache(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.reset_cache()

    def reset_cache(self):
        self.k_cache = None
        self.v_cache = None
        self.kv_computations = 0  # how many tokens' K/V we've EVER computed

    def _split_heads(self, x, batch_size, seq_len):
        return x.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

    def step(self, x_new):
        # x_new: (batch, 1, d_model) — just the newest token, nothing else
        batch_size = x_new.shape[0]

        Q = self._split_heads(self.W_q(x_new), batch_size, 1)
        K_new = self._split_heads(self.W_k(x_new), batch_size, 1)
        V_new = self._split_heads(self.W_v(x_new), batch_size, 1)
        self.kv_computations += 1  # only ONE token's worth of K/V, ever, per step

        if self.k_cache is None:
            self.k_cache, self.v_cache = K_new, V_new
        else:
            self.k_cache = torch.cat([self.k_cache, K_new], dim=2)  # grow along seq_len
            self.v_cache = torch.cat([self.v_cache, V_new], dim=2)

        # new token's query attends over ALL cached keys/values, past + itself
        output, weights = scaled_dot_product_attention(Q, self.k_cache, self.v_cache)
        output = output.transpose(1, 2).contiguous().view(batch_size, 1, -1)
        return self.W_o(output), weights

**Key line**: ```Q``` has shape ```(..., 1, head_dim)``` — just the new token — but ```self.k_cache```/```self.v_cache``` have shape ```(..., cache_len, head_dim)``` — every token so far. ```scaled_dot_product_attention``` doesn't care that ```Q``` and ```K``` have different sequence lengths; the matrix multiply just produces a ```(1, cache_len)``` attention row instead of a square matrix. That asymmetry is the KV-cache — one new query, checked against everything remembered.

Notice this also gets causality for free: the cache can only ever contain past tokens (future ones don't exist yet during generation), so there's no risk of a token attending to something that hasn't been generated yet.

### Compare: cached vs. naive token-by-token generation

Same 6 toy "tokens," generated one at a time. Count how many K/V vectors
each approach computes, in total, across the whole generation.

In [ ]:
torch.manual_seed(0)
d_model, num_heads, batch_size, n_steps = 8, 2, 1, 6

cached_mha = MultiHeadAttentionWithCache(d_model, num_heads)
tokens = [torch.randn(batch_size, 1, d_model) for _ in range(n_steps)]

naive_kv_computations = 0
for t, tok in enumerate(tokens, start=1):
    cached_mha.step(tok)
    naive_kv_computations += t  # naive recomputes K/V for all t tokens, every step

print("Cached total K/V computations:", cached_mha.kv_computations)
print("Naive total K/V computations:", naive_kv_computations)

## **Causal Masking**

**Why it matters:** during training we feed the model a whole sentence at
once and ask it to predict every next word simultaneously. If attention
lets token 3 look at token 5, it can just copy the answer instead of
learning to predict it. A causal mask blocks every token from seeing
anything that comes after it.

**ELI5:** imagine reading a sentence left to right with your hand covering
everything past the word you're on. You can see what you've already read
plus the current word, but nothing ahead — so you can't peek at the answer
when guessing the next word.

In [ ]:
def causal_mask(seq_len):
    """seq_len x seq_len mask: token i can see tokens 0..i, nothing after."""
    return torch.tril(torch.ones(seq_len, seq_len)).bool()

causal_mask(4)

### Testing the mask, without touching transformer.py yet

A standalone copy of the attention function with masking added, so we can
verify the mechanism in isolation before merging it into the real file.

In [ ]:
def scaled_dot_product_attention_masked(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

- ```scores.masked_fill(mask == 0, float('-inf'))``` — wherever the mask is ```0```/```False```, overwrite that score with negative infinity, before softmax.
- ```softmax(-inf) = 0```, so that position gets exactly zero attention weight.

In [ ]:
torch.manual_seed(0)
seq_len, d_k = 4, 8
Q = torch.randn(seq_len, d_k)
K = torch.randn(seq_len, d_k)
V = torch.randn(seq_len, d_k)

mask = causal_mask(seq_len)
output_masked, weights_masked = scaled_dot_product_attention_masked(Q, K, V, mask=mask)

print("masked weights:\n", weights_masked)
print("row sums:", weights_masked.sum(dim=-1))

### The full transformer.py

In [ ]:
%%writefile src/transformer.py
import torch
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights


def causal_mask(seq_len):
    """seq_len x seq_len mask: token i can see tokens 0..i, nothing after."""
    return torch.tril(torch.ones(seq_len, seq_len)).bool()


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        output, weights = scaled_dot_product_attention(Q, K, V, mask=mask)

        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        return self.W_o(output), weights


class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        return self.norm(x + sublayer_output)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.activation(self.linear1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)

    def forward(self, x, mask=None):
        attn_output, weights = self.mha(x, mask=mask)
        x = self.add_norm1(x, attn_output)

        ff_output = self.ff(x)
        x = self.add_norm2(x, ff_output)

        return x, weights


class MultiHeadAttentionWithCache(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.reset_cache()

    def reset_cache(self):
        self.k_cache = None
        self.v_cache = None
        self.kv_computations = 0

    def _split_heads(self, x, batch_size, seq_len):
        return x.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

    def step(self, x_new):
        batch_size = x_new.shape[0]

        Q = self._split_heads(self.W_q(x_new), batch_size, 1)
        K_new = self._split_heads(self.W_k(x_new), batch_size, 1)
        V_new = self._split_heads(self.W_v(x_new), batch_size, 1)
        self.kv_computations += 1

        if self.k_cache is None:
            self.k_cache, self.v_cache = K_new, V_new
        else:
            self.k_cache = torch.cat([self.k_cache, K_new], dim=2)
            self.v_cache = torch.cat([self.v_cache, V_new], dim=2)

        output, weights = scaled_dot_product_attention(Q, self.k_cache, self.v_cache)
        output = output.transpose(1, 2).contiguous().view(batch_size, 1, -1)
        return self.W_o(output), weights

In [ ]:
import importlib
import transformer
importlib.reload(transformer)
import model
importlib.reload(model)

from transformer import causal_mask, TransformerBlock
from model import TokenAndPositionalEmbedding, ToyLanguageModel

## **Step 2 - Embeddings (Token + Positional)**

**Why it matters:** the tokenizer outputs integers (token IDs). Every
transformer piece we've built expects vectors, not integers. An embedding
is a lookup table — one learned vector per token ID.

There's a second problem, though: attention on its own has no sense of
*order*. Shuffle the tokens going in, and attention just gives you the same
outputs, shuffled the same way — it only cares about content, not position.
So we add a second lookup table, indexed by *position* (1st word, 2nd word,
...), and add it directly onto the token embedding.

**ELI5:** the token embedding is like giving each word a "meaning
fingerprint" pulled from a big table. But if that's all you had, the model
couldn't tell "cat sat mat" apart from "mat sat cat" — same words, same
fingerprints, attention doesn't care about order on its own. The positional
embedding is a second stamp on each word saying "you're the 1st word,"
"you're the 2nd word," etc., so position gets baked in right alongside
meaning.

In [ ]:
class TokenAndPositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)
        return self.token_embedding(token_ids) + self.position_embedding(positions)

- ```nn.Embedding(vocab_size, d_model)``` — literally a matrix of shape ```(vocab_size, d_model)```: one learned row per possible token ID. ```self.token_embedding(token_ids)``` looks up each ID's row.

- ```nn.Embedding(max_seq_len, d_model)``` — same mechanism, but the "ID" here is a position (0, 1, 2, ...) instead of a token identity. Row 0 always means "I'm the first token in the sequence," regardless of which word actually sits there.

- ```torch.arange(seq_len).unsqueeze(0)``` — builds ```[0, 1, 2, ..., seq_len-1]```, then adds a batch dimension so it lines up with ```token_ids```'s shape.

- Adding the two embeddings together ```(elementwise)``` merges "what this token is" and "where it sits" into one ```d_model-sized``` vector per position — exactly the shape ```TransformerBlock``` expects as input.

In [ ]:
torch.manual_seed(0)
vocab_size, max_seq_len, d_model = 20, 10, 8
embed = TokenAndPositionalEmbedding(vocab_size, max_seq_len, d_model)

token_ids = torch.tensor([[3, 7, 1, 19, 4]])  # batch=1, seq_len=5
x = embed(token_ids)
print("shape:", x.shape)  # expect (1, 5, 8)

## **Step 3 - The Full Model**

Stack the pieces: embed tokens → pass through several `TransformerBlock`s in
sequence, each one refining the representation further → project the final
vectors into a score for every possible next token.

**ELI5:** several "committees" (transformer blocks) work on the sentence one
after another, each committee refining what the previous one produced. After
the last committee, for every position you ask: "given everything you now
know, what's your best guess for the token that comes right after this
one?" — that guess is one score per possible token in the vocabulary (a
"logit"), not yet a probability.

In [ ]:
%%writefile src/model.py
import torch
import torch.nn as nn

from transformer import causal_mask, TransformerBlock


class TokenAndPositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)
        return self.token_embedding(token_ids) + self.position_embedding(positions)


class ToyLanguageModel(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding = TokenAndPositionalEmbedding(vocab_size, max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        x = self.embedding(token_ids)
        mask = causal_mask(seq_len)

        attn_weights_per_layer = []
        for block in self.blocks:
            x, weights = block(x, mask=mask)
            attn_weights_per_layer.append(weights)

        logits = self.output_head(x)
        return logits, attn_weights_per_layer

- ```nn.ModuleList([...])``` — not a plain Python list. This matters: PyTorch only tracks parameters (for the optimizer, ```.to(device```), saving/loading) on submodules registered through things like ```nn.ModuleList```. A plain ```[TransformerBlock(...), ...]``` would silently hide all those layers' weights from training.

- ```mask = causal_mask(seq_len)``` computed fresh inside forward — so the model works for whatever sequence length gets passed in, not a fixed size baked in at construction.

- The loop passes ```x``` through each block in sequence, each one updating it further, and we collect every layer's attention weights along the way — that's what lets us "inspect" attention later, per the deliverable.

- ```self.output_head(x)```: ```Linear(d_model, vocab_size)``` turns each token's final vector into one score per vocabulary entry. These are called **logits** — raw scores, not yet probabilities (that conversion happens in the loss function during training, and during sampling at generation time).

In [ ]:
import importlib
import model
importlib.reload(model)
from model import TokenAndPositionalEmbedding, ToyLanguageModel

In [ ]:
# testing the full code
torch.manual_seed(0)
vocab_size, max_seq_len, d_model, num_heads, d_ff, num_layers = 20, 10, 8, 2, 32, 2
model = ToyLanguageModel(vocab_size, max_seq_len, d_model, num_heads, d_ff, num_layers)

token_ids = torch.tensor([[3, 7, 1, 19, 4]])
logits, attn_weights = model(token_ids)

print("logits shape:", logits.shape)                      # expect (1, 5, 20)
print("number of layers:", len(attn_weights))              # expect 2
print("layer 0 attention weights shape:", attn_weights[0].shape)  # expect (1, 2, 5, 5)

### Save your progress

Colab's runtime is ephemeral — anything not pushed disappears when it resets.

In [ ]:
!git pull origin main --no-rebase --no-edit #weirdly this is needed because saving to github via colab causes issues.
!git add -A
!git commit -m "Module 0 updates"
!git push

## Final Set Up

By the end of Module 0, the folder structure should look like this:

```
learning_llm/
|- notebooks/
│  |- 00_mechanics.ipynb
|- src/
│  |- tokenizer.py
|  |- transformer.py
|  |- models.py
```

The code below would be the final code needed. You would already have them above.

In [ ]:
# Module 0 - Tokenizer Code
# src/tokenizer.py

from collections import defaultdict

def get_pair_counts(word_freqs):
    pair_counts = defaultdict(int)
    for word, freq in word_freqs.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pair_counts[pair] += freq
    return pair_counts


def merge_pair(pair, word_freqs):
    new_word_freqs = {}
    for word, freq in word_freqs.items():
        new_word = []
        i = 0
        while i < len(word):
            if i < len(word) - 1 and (word[i], word[i + 1]) == pair:
                new_word.append(word[i] + word[i + 1])
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_freqs[tuple(new_word)] = freq
    return new_word_freqs


def train_bpe(word_freqs, num_merges):
    word_freqs = dict(word_freqs)
    merges = []
    for step in range(num_merges):
        pair_counts = get_pair_counts(word_freqs)
        if not pair_counts:
            break
        best_pair = max(pair_counts, key=pair_counts.get)
        word_freqs = merge_pair(best_pair, word_freqs)
        merges.append(best_pair)
        print(f"Merge {step + 1}: {best_pair}  (count={pair_counts[best_pair]})")
    return word_freqs, merges


def get_word_tokens(word, merges):
    """Apply a learned merge list, in order, to a single word."""
    tokens = list(word) + ['</w>']
    for pair in merges:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
                new_tokens.append(tokens[i] + tokens[i + 1])
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return tokens


def encode(text, merges):
    """Whitespace-split text into words, then BPE-tokenize each word."""
    all_tokens = []
    for word in text.strip().split():
        all_tokens.extend(get_word_tokens(word, merges))
    return all_tokens


def decode(tokens):
    """Join tokens back into text."""
    text = ''.join(tokens).replace('</w>', ' ')
    return text.strip()

In [ ]:
# Module 0 - Transformer Code
# src/transformer.py


import torch
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights


def causal_mask(seq_len):
    """seq_len x seq_len mask: token i can see tokens 0..i, nothing after."""
    return torch.tril(torch.ones(seq_len, seq_len)).bool()


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape

        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        output, weights = scaled_dot_product_attention(Q, K, V, mask=mask)

        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        return self.W_o(output), weights


class ResidualLayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, sublayer_output):
        return self.norm(x + sublayer_output)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.linear2(self.activation(self.linear1(x)))


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.add_norm1 = ResidualLayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff)
        self.add_norm2 = ResidualLayerNorm(d_model)

    def forward(self, x, mask=None):
        attn_output, weights = self.mha(x, mask=mask)
        x = self.add_norm1(x, attn_output)

        ff_output = self.ff(x)
        x = self.add_norm2(x, ff_output)

        return x, weights


class MultiHeadAttentionWithCache(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.reset_cache()

    def reset_cache(self):
        self.k_cache = None
        self.v_cache = None
        self.kv_computations = 0

    def _split_heads(self, x, batch_size, seq_len):
        return x.view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

    def step(self, x_new):
        batch_size = x_new.shape[0]

        Q = self._split_heads(self.W_q(x_new), batch_size, 1)
        K_new = self._split_heads(self.W_k(x_new), batch_size, 1)
        V_new = self._split_heads(self.W_v(x_new), batch_size, 1)
        self.kv_computations += 1

        if self.k_cache is None:
            self.k_cache, self.v_cache = K_new, V_new
        else:
            self.k_cache = torch.cat([self.k_cache, K_new], dim=2)
            self.v_cache = torch.cat([self.v_cache, V_new], dim=2)

        output, weights = scaled_dot_product_attention(Q, self.k_cache, self.v_cache)
        output = output.transpose(1, 2).contiguous().view(batch_size, 1, -1)
        return self.W_o(output), weights

In [ ]:
# Module 0 - Embeddings / Model Code
# src/model.py

import torch
import torch.nn as nn

from transformer import causal_mask, TransformerBlock


class TokenAndPositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)
        return self.token_embedding(token_ids) + self.position_embedding(positions)


class ToyLanguageModel(nn.Module):
    def __init__(self, vocab_size, max_seq_len, d_model, num_heads, d_ff, num_layers):
        super().__init__()
        self.embedding = TokenAndPositionalEmbedding(vocab_size, max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.output_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        batch_size, seq_len = token_ids.shape
        x = self.embedding(token_ids)
        mask = causal_mask(seq_len)

        attn_weights_per_layer = []
        for block in self.blocks:
            x, weights = block(x, mask=mask)
            attn_weights_per_layer.append(weights)

        logits = self.output_head(x)
        return logits, attn_weights_per_layer

In [ ]:
# Required Import

import importlib
import tokenizer
importlib.reload(tokenizer)
from tokenizer import get_pair_counts, merge_pair, train_bpe, encode, decode
import transformer
importlib.reload(transformer)
from transformer import (
    scaled_dot_product_attention,
    causal_mask,
    MultiHeadAttention,
    ResidualLayerNorm,
    FeedForward,
    TransformerBlock,
    MultiHeadAttentionWithCache,
)
import model
importlib.reload(model)
from model import TokenAndPositionalEmbedding, ToyLanguageModel


In [ ]:
!git pull origin main --no-rebase --no-edit
!git add -A
!git commit -m "Move code into src/"
!git push